*Workflow*
Load train images -> Normalize -> Augment -> Train
Load test images -> Normalize -> Evaluate

In [2]:
from PIL import Image
import numpy as np
import os

def load_images(folder):
    images = []
    labels = []
    
    class_names = sorted(os.listdir(folder))
    
    for label, class_name in enumerate(class_names):
        class_path = os.path.join(folder, class_name)
        
        if not os.path.isdir(class_path):
            continue
        
        for file in os.listdir(class_path):
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                path = os.path.join(class_path, file)
                
                img = Image.open(path)
                img = np.array(img, dtype=np.float32)
                
                images.append(img)
                labels.append(label)
    
    images = np.array(images)
    labels = np.array(labels)
    
    return images, labels

In [3]:
train_dir = r"cross_val_folds/fold_1/train"
test_dir = r"cross_val_folds/fold_1/test"

# Load images and labels
X_train, y_train = load_images(train_dir)
X_test, y_test = load_images(test_dir)

# Normalize image data
X_train = X_train / 255.0
X_test = X_test / 255.0

# Add channel dimension
X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

In [ ]:
import random

def augment_image(img):
    # Horizontal flip
    if random.random() > 0.5:
        img = np.fliplr(img)
    
    # Vertical flip
    if random.random() > 0.5:
        img = np.flipud(img)
    
    # Random rotation (90° steps for simplicity)
    k = random.randint(0, 3)
    img = np.rot90(img, k)
    
    return img

# Apply augmentation to training images
X_train_augmented = np.array([augment_image(img) for img in X_train])

# Combine original and augmented data
X_train = np.concatenate([X_train, X_train_augmented])
y_train = np.concatenate([y_train, y_train])
print(X_train.min(), X_train.max())

0.0 1.0
